In [ ]:
!unzip -o /tmp/images.zip -d /tmp/

unzip:  cannot find or open /tmp/images.zip, /tmp/images.zip.zip or /tmp/images.zip.ZIP.


In [ ]:
!unzip -q images.zip

unzip:  cannot find or open images.zip, images.zip.zip or images.zip.ZIP.


In [ ]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: /content
Files here: ['.config', 'images.zip', 'sample_data']


In [ ]:
!mv /images.zip /content/

In [ ]:
!unzip -q images.zip

replace images/.ipynb_checkpoints/Zoology.pdf-0021-01-checkpoint.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
print("Images found:", len(os.listdir("images")))

FileNotFoundError: [Errno 2] No such file or directory: 'images'

In [ ]:

!pip install -q transformers accelerate qwen-vl-utils torch

import os
import json
import time
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# 2. Load the Model
print("Loading Qwen2.5-VL-3B... (This takes a minute or two on the first run)")
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

image_folder = "images"
output_json = "qwen_reference_captions.json"
output_md = "qwen_reference_captions.md"


PROMPT = """Look at this image extracted from a textbook.

If it is a real diagram or illustration, write a concise 1-2 sentence caption that acts as a quick reference.

Your caption MUST include:
1. The exact biological subject (e.g., "Human digestive system", "Dental formula", "Alveolus gas exchange").
2. The specific anatomical parts or labels shown (e.g., "showing the fundus, cardiac region, and pyloric sphincter").

Do NOT describe the colors, shapes, or background. Do not summarize the surrounding text. Keep it factual and brief.
If it is a fragment (a single word, a tiny line, or meaningless shapes), reply exactly with: FRAGMENT"""

print(f"\nProcessing {len(os.listdir(image_folder))} images with Qwen...")
files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(".png")])
dataset = []

for fname in files:
    img_path = os.path.join(image_folder, fname)
    print(f"Checking {fname}..", end=" ")

    # 4. Format the input for Qwen
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    # Prepare inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # 5. Generate the caption
    try:
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=150)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        caption = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        # 6. Filter and Save
        if "FRAGMENT" in caption:
            print("✗ Fragment/Junk (Skipped)")
        else:
            print("✓ Valid Diagram")
            dataset.append({
                "image_file": fname,
                "caption": caption
            })

    except Exception as e:
        print(f"✗ Error: {e}")

    time.sleep(1)

# --- 7. Save JSON ---
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print(f"\n✓ Saved data to {output_json}")

# --- 8. Save Markdown File ---
with open(output_md, "w", encoding="utf-8") as f:
    f.write("# Zoology Image Reference (Qwen 3B)\n\n")
    f.write(f"Total valid diagrams extracted: **{len(dataset)}**\n\n")
    f.write("---\n\n")

    for item in dataset:
        f.write(f"### {item['image_file']}\n\n")
        f.write(f"![{item['image_file']}]({image_folder}/{item['image_file']})\n\n")
        f.write(f"**Caption:** {item['caption']}\n\n")
        f.write("---\n\n")

print(f"✓ Saved visual markdown file to {output_md}")

Loading Qwen2.5-VL-3B... (This takes a minute or two on the first run)


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]


Processing 30 images with Qwen...
Checking Zoology.pdf-0021-01.png.. ✓ Valid Diagram
Checking Zoology.pdf-0023-05.png.. ✗ Error: CUDA out of memory. Tried to allocate 874.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 659.81 MiB is free. Including non-PyTorch memory, this process has 13.92 GiB memory in use. Of the allocated memory 13.75 GiB is allocated by PyTorch, and 44.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Checking Zoology.pdf-0024-04.png.. ✗ Fragment/Junk (Skipped)
Checking Zoology.pdf-0024-05.png.. ✗ Fragment/Junk (Skipped)
Checking Zoology.pdf-0024-06.png.. ✗ Fragment/Junk (Skipped)
Checking Zoology.pdf-0024-07.png.. ✗ Fragment/Junk (Skipped)
Checking Zoology.pdf-0024-08.png.. ✗ Fragme

In [ ]:
!find / -name "images.zip" 2>/dev/null

/var/images.zip
/content/images.zip


In [ ]:
from google.colab import files
import os

if os.path.exists("images.zip"):
    os.remove("images.zip")


uploaded = files.upload()
print("Upload complete!")

Click the button below and select images.zip from your computer:


Saving images.zip to images.zip
Upload complete!


In [ ]:
!pip install llama_cloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 6.2 MB/s  0:00:00


In [ ]:
!pip install llama-cloud


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 6.1 MB/s  0:00:00


In [ ]:
from google.colab import files
uploaded=files.upload()
print("files uploaded")


Saving physics-2.pdf to physics-2 (1).pdf
files uploaded


In [ ]:
from pathlib import Path
from llama_cloud import LlamaCloud
client=LlamaCloud(api_key="llx-hZuwnemZFsOfVbYsUi6kwwNBLO43AUK4rIxUkdgfUu61RhQp")
file_obj=client.files.create(
    file=Path("physics-2.pdf"),
    purpose="parse",
    )
print(file_obj)
result=client.parsing.create(
    tier="agentic",
    version="latest",
    file_id=file_obj.id,
    page_ranges={
        "max_pages":25
    }

)

print(result.id)



FileCreateResponse(id='abfa2d1e-7ba3-44f1-83c3-466d6cad15b5', name='physics-2.pdf', project_id='b704deed-50ae-4ecc-b7e4-237120264425', download_url=None, expires_at=datetime.datetime(2026, 10, 30, 8, 12, 33, 676867, tzinfo=TzInfo(0)), external_file_id='096d42a2-053d-4f1f-879e-fc32b9e832d5', file_type='pdf', last_modified_at=None, purpose='parse')
pjb-ts5fgp9fiamxvh0qiiy3q0b76ipm


In [ ]:
import llama_cloud
print(llama_cloud.__version__)

2.19.0


In [ ]:
import time
job_id="pjb-ts5fgp9fiamxvh0qiiy3q0b76ipm"
result=client.parsing.get(job_id, expand=["markdown"])
if result.job.status=="COMPLETED":
  md_content=""
  for i, page in enumerate(result.markdown.pages):
    if i>=25:
      break
    md_content=md_content+page.markdown+ "\n\n"

  with open("physics.md", "w", encoding="UTF-8") as f:
    f.write(md_content)
  print("File saved successfully")

else:
  print("Status", result.job.status)
  time.sleep(5)






File saved successfully


In [ ]:
result = client.parsing.get("pjb-mj2i5mdg5ug0q4okhhx0vt9lzuj7", expand=["markdown"])
print(result.job.status)


COMPLETED


In [ ]:
!pip install -q marker-pdf

In [ ]:
!marker_single physics-2.pdf --output_dir ./output

Traceback (most recent call last):
  File "/usr/local/bin/marker_single", line 6, in <module>
    sys.exit(convert_single_cli())
             ~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1631, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1552, in main
    rv = self.invoke(ctx)
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1415, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 910, in invoke
    return callback(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/marker/scripts/convert_single.py", line 38, in convert_single_cli
    rendered = converter(fpath)
  File "/usr/local/lib/python3.13/dist-packages/marker/converters/pdf.py", line 213, in __call__
    document =

In [ ]:

with open("physics.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()


print("Display formulas ($$):", markdown_text.count("$$") // 2)
print("Inline $ markers:", markdown_text.count("$"))



Display formulas ($$): 191
Inline $ markers: 1746


In [ ]:
with open("physics.md", "r", encoding="utf-8") as f:
    raw = f.read()

print("chars:", len(raw))
print("$$ markers:", raw.count("$$"))

chars: 82594
$$ markers: 382


In [ ]:
def clean_latex(md):

    out, i = [], 0
    while i < len(md):
        if md[i:i+5] == "\\tag{":
            end = md.find("}", i)
            if end != -1:
                i = end + 1
                continue
        out.append(md[i])
        i += 1
    md = "".join(out)


    lines = []
    for ln in md.split("\n"):
        s = ln.rstrip()
        if s.endswith(")") and "$$" in s:
            idx = s.rfind("$$")
            tail = s[idx+2:].strip()
            if tail.startswith("(") and tail.endswith(")") and tail[1:-1].isdigit():
                ln = s[:idx+2]
        if "$$" in ln and ln.rstrip().endswith("$$"):
            idx = ln.find("$$")
            before = ln[:idx].rstrip()
            if before.endswith((",", ";", ".")):
                ln = before.rstrip(",;.").rstrip() + ln[idx:]
        lines.append(ln)
    return "\n".join(lines)

md = clean_latex(raw)
print("after clean — $$:", md.count("$$"))
print("chars:", len(md))

after clean — $$: 382
chars: 82594


In [ ]:
def drop_boilerplate(md):
    stop = ("references", "bibliography", "appendix", "acknowledg")
    kept = []
    for ln in md.split("\n"):
        s = ln.strip().lower()
        if s.startswith("#") and any(w in s for w in stop):
            break
        kept.append(ln)
    md = "\n".join(kept)

    md = "\n".join(ln for ln in md.split("\n") if not ln.strip().isdigit())
    while "\n\n\n" in md:
        md = md.replace("\n\n\n", "\n\n")
    return md

md = drop_boilerplate(md)
print("after boilerplate — chars:", len(md))
print("still has 'references' (lowercase, anywhere):", "references" in md.lower())
print("$$ markers:", md.count("$$"))

after boilerplate — chars: 82594
still has 'references' (lowercase, anywhere): False
$$ markers: 382


In [ ]:
def clean_latex(md):

    out, i = [], 0
    while i < len(md):
        if md[i:i+5] == "\\tag{":
            end = md.find("}", i)
            if end != -1:
                i = end + 1
                continue
        out.append(md[i])
        i += 1
    md = "".join(out)

    lines = []
    for ln in md.split("\n"):
        s = ln.rstrip()
        if s.endswith(")") and "$$" in s:
            idx = s.rfind("$$")
            tail = s[idx+2:].strip()
            if tail.startswith("(") and tail.endswith(")") and tail[1:-1].isdigit():
                ln = s[:idx+2]
        if "$$" in ln and ln.rstrip().endswith("$$"):
            idx = ln.find("$$")
            before = ln[:idx].rstrip()
            if before.endswith((",", ";", ".")):
                ln = before.rstrip(",;.").rstrip() + ln[idx:]
        lines.append(ln)
    return "\n".join(lines)

def drop_boilerplate(md):

    stop = ("references", "bibliography", "appendix", "acknowledg")
    kept = []
    for ln in md.split("\n"):
        s = ln.strip().lower()
        if s.startswith("#") and any(w in s for w in stop):
            break
        kept.append(ln)
    md = "\n".join(kept)

    md = "\n".join(ln for ln in md.split("\n") if not ln.strip().isdigit())

    while "\n\n\n" in md:
        md = md.replace("\n\n\n", "\n\n")
    return md

with open("physics.md", "r", encoding="utf-8") as f:
    raw = f.read()

md_clean = clean_latex(raw)
md_clean = drop_boilerplate(md_clean)

with open("physics-2_preprocessed.md", "w", encoding="utf-8") as f:
    f.write(md_clean)

print("saved physics-2_preprocessed.md")
print("chars:", len(md_clean))
print("$$ markers:", md_clean.count("$$"))

saved physics-2_preprocessed.md
chars: 82594
$$ markers: 382


In [ ]:
from bs4 import BeautifulSoup
def html_tables_to_markdown(md):
    soup = BeautifulSoup(md, "html.parser")
    for table in soup.find_all("table"):
        rows = table.find_all("tr")
        if not rows:
            continue
        lines = []
        for i, row in enumerate(rows):
            cells = row.find_all(["th", "td"])
            row_text = [c.get_text(strip=True) for c in cells]
            if i == 0:
                lines.append("| " + " | ".join(row_text) + " |")
                lines.append("|" + "---|" * len(row_text))
            else:
                lines.append("| " + " | ".join(row_text) + " |")
        table.replace_with("\n".join(lines))
    return str(soup)

In [ ]:
def sup_to_latex(md):
    out, i = [], 0
    while i < len(md):
        if md[i:i+5] == "<sup>":
            end = md.find("</sup>", i)
            if end != -1:
                out.append("^{" + md[i+5:end] + "}")
                i = end + 6
                continue
        out.append(md[i])
        i += 1
    return "".join(out)

In [ ]:
with open("physics-2_converted.md", "w", encoding="utf-8") as f:
    f.write(md)

print("saved physics-2_converted.md")
print("chars:", len(md))
print("$$:", md.count("$$"))

saved physics-2_converted.md
chars: 80999
$$: 382


In [ ]:

bad_start = md.find("$$a = a_{mean} \\pm \\Delta a_{mean}  which")
bad_end = md.find("**relative error**.$$", bad_start)
if bad_start != -1 and bad_end != -1:
    bad_end += len("**relative error**.$$")
    replacement = (
        "$$a = a_{mean} \\pm \\Delta a_{mean}$$\n\n"
        "which tells us that the actual value of $a$ could be between "
        "$a_{mean} - \\Delta a_{mean}$ and $a_{mean} + \\Delta a_{mean}$. "
        "The ratio of mean absolute error to its arithmetic mean value is "
        "called **relative error**."
    )
    md = md[:bad_start] + replacement + md[bad_end:]
    print("1. fixed broken formula")


out, i = [], 0
while i < len(md):
    if md[i:i+5] == "<sub>":
        end = md.find("</sub>", i)
        if end != -1:
            out.append("_{" + md[i+5:end] + "}")
            i = end + 6
            continue
    if md[i:i+3] == "<u>":
        i += 3
        continue
    if md[i:i+4] == "</u>":
        i += 4
        continue
    out.append(md[i])
    i += 1
md = "".join(out)
print("2. fixed <u> and <sub>")


lines = []
for ln in md.split("\n"):
    s = ln.strip().lower()
    if s.startswith(("drawing:", "icon:", "flow_chart:", "logo:")):
        continue
    lines.append(ln)
md = "\n".join(lines)
print("3. dropped placeholders")


lines, in_fence = [], False
for ln in md.split("\n"):
    if ln.strip().startswith("```"):
        in_fence = not in_fence
        continue
    if not in_fence:
        lines.append(ln)
md = "\n".join(lines)
print("4. dropped mermaid blocks")

1. fixed broken formula
2. fixed <u> and <sub>
3. dropped placeholders
4. dropped mermaid blocks


In [ ]:
import re
md = re.sub(r'\$\$(.*?)---[^$]*?\$\$', r'$$\1$$', md, flags=re.DOTALL)

lines = []
for ln in md.split("\n"):
    if "---" in ln and "$$" in ln:

        idx = ln.rfind("$$")
        ln = ln[:idx+2]
    lines.append(ln)
md = "\n".join(lines)


out = []
i = 0
while i < len(md):
    if md[i:i+3] == "---":

        j = i + 3
        while j < len(md) and md[j] == " ":
            j += 1
        if j < len(md) and md[j] == "(":
            k = md.find(")", j)
            if k != -1 and md[j+1:k].strip().isdigit():
                i = k + 1
                continue
    out.append(md[i])
    i += 1
md = "".join(out)

print("---(N) left:", md.count("--- ("))
print("--- inside $$ left:", md.count("--- from"))
print("$$:", md.count("$$"))

---(N) left: 5
--- inside $$ left: 0
$$: 382


In [ ]:
with open("physics_cleanedfinal.md", "w", encoding="utf-8") as f:
    f.write(md)

print("physics_cleanedfinal.md")
print("chars:", len(md))
print("$$:", md.count("$$"))

physics_cleanedfinal.md
chars: 76342
$$: 382


In [8]:
import re

path = "physics-2_converted.md"

with open(path, "r", encoding="utf-8") as f:
    md = f.read()

before_chars = len(md)
before_dd = md.count("$$")


md = md.replace("<u>", "").replace("</u>", "")


out, i = [], 0
while i < len(md):
    if md[i:i+5] == "<sub>":
        end = md.find("</sub>", i)
        if end != -1:
            out.append("_{" + md[i+5:end] + "}")
            i = end + 6
            continue
    out.append(md[i])
    i += 1
md = "".join(out)


lines = []
for ln in md.split("\n"):
    s = ln.strip().lower()
    if s.startswith(("drawing:", "icon:", "flow_chart:", "logo:")):
        continue
    lines.append(ln)
md = "\n".join(lines)


lines, in_fence = [], False
for ln in md.split("\n"):
    if ln.strip().startswith("```"):
        in_fence = not in_fence
        continue
    if not in_fence:
        lines.append(ln)
md = "\n".join(lines)


def fix_dim(m):
    inner = m.group(1)
    parts = re.findall(r'([LMT])(-?\d+)', inner)
    if len(parts) == 3:
        return "[" + "".join(f"{s}^{{{e}}}" for s, e in parts) + "]"
    return m.group(0)

md = re.sub(r'\[([LMT]-?\d+[LMT]-?\d+[LMT]-?\d+)\]', fix_dim, md)


bad_start = md.find("$$a = a_{mean} \\pm \\Delta a_{mean}  which")
bad_end = md.find("**relative error**.$$", bad_start)
if bad_start != -1 and bad_end != -1:
    bad_end += len("**relative error**.$$")
    replacement = (
        "$$a = a_{mean} \\pm \\Delta a_{mean}$$\n\n"
        "which tells us that the actual value of $a$ could be between "
        "$a_{mean} - \\Delta a_{mean}$ and $a_{mean} + \\Delta a_{mean}$. "
        "The ratio of mean absolute error to its arithmetic mean value is "
        "called **relative error**."
    )
    md = md[:bad_start] + replacement + md[bad_end:]
    print("fixed broken formula 1.8.1")
else:
    print("broken formula already OK")


md = re.sub(r'\s*-{2,4}\s*\(\s*\d+(\.\d+)?\s*\)', '', md)


with open(path, "w", encoding="utf-8") as f:
    f.write(md)

print("saved:", path)


FileNotFoundError: [Errno 2] No such file or directory: 'physics-2_converted.md'

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter

with open("physics-2_converted.md", "r", encoding="utf-8") as f:
    text = f.read()

splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = splitter.create_documents([text])
chunks = [d.page_content for d in docs]

print(f"Total chunks: {len(chunks)}")
print(f"total $$ in chunks: {sum(c.count('$$') for c in chunks)}")
bad = [i for i, c in enumerate(chunks) if c.count("$$") % 2 != 0]
print(f"chunks with unclosed $$: {bad}")

Total chunks: 44
total $$ in chunks: 418
chunks with unclosed $$: []


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter
import json


with open("physics-2_converted.md", "r", encoding="utf-8") as f:
    text = f.read()


splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = splitter.create_documents([text])
chunks = [d.page_content for d in docs]

fixed, carry = [], ""
for c in chunks:
    c = carry + c
    if c.count("$$") % 2 != 0:
        idx = c.rfind("$$")
        carry = c[idx:]
        c = c[:idx]
    else:
        carry = ""
    if c.strip():
        fixed.append(c.strip())
if carry:
    fixed.append(carry)
chunks = fixed

data = [{"id": i + 1, "text": c} for i, c in enumerate(chunks)]

with open("physics_chunks.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"saved physics_chunks.json with {len(data)} chunks")
print(f"total $$ in chunks: {sum(c.count('$$') for c in chunks)}")
print(f"unclosed $$ chunks: {[i for i, c in enumerate(chunks) if c.count('$$') % 2 != 0]}")

saved physics_chunks.json with 44 chunks
total $$ in chunks: 418
unclosed $$ chunks: []


In [2]:
!pip install -q google-genai

In [6]:
import os, json, time
from google import genai
from typing import Literal
from pydantic import BaseModel, Field
from google.genai import types

client = genai.Client(GEMINI_API_KEY)

SYSTEM_PROMPT = """You are an expert exam question writer for a Physics textbook.
Create high-quality MCQs based ONLY on the source text provided.

1. SOURCE GROUNDING
Every question, answer and distractor must be supported by the source.
Do not introduce outside knowledge. If the source does not support a
question, do not generate it.

2. HOW MANY QUESTIONS — BE STRICT
Generate AT MOST 3 questions per chunk.
If the chunk has fewer than 2 clearly testable concepts, generate only 1.
If the chunk is a heading, table of contents, figure caption, or has no
testable concept, return {"mcqs": []}.

Do NOT generate a question just because there is text.
Do NOT pad to reach 3.

3. EVERY QUESTION MUST NAME A SPECIFIC CONCEPT
The question must be about a named, identifiable concept from the source:
a specific quantity, unit, law, formula, definition, structure, process,
or relationship.

BAD (vague — do NOT write these):
- "Which of the following is correct about measurement?"
- "What can be said about units?"
- "Which statement about errors is true?"
- "What is important in dimensional analysis?"

GOOD (specific — write these):
- "What is the SI unit of velocity?"
- "Which formula gives the dimensional formula of force?"
- "Why is the parallax method used to measure stellar distances?"
- "What is the least count of a metre scale?"

If you cannot name the specific concept in one short phrase, do not
write the question.

4. FORMULAS
Any formula in a question or option must be LaTeX:
inline $...$, display $$...$$. Never plain text, never Unicode symbols.

5. OPTIONS
Exactly 4 options per question, exactly one correct.
All 4 options must belong to the same conceptual category.
Wrong options must be plausible and source-grounded.

6. ALIGNMENT
The question must ask exactly what the correct answer provides.
Do not ask "why" when the answer is a definition.

7. WORDING — BANNED PHRASES
Never write any of these:
- "according to the text"
- "according to the passage"
- "mentioned in the text"
- "mentioned in the passage"
- "as stated in the text"
- "from the given text"
- "based on the passage"
- "the text says"

Write the question as if it stands on its own.

8. DIFFICULTY
EASY = direct recall. MEDIUM = distinguishing related concepts.
HARD = applying or synthesizing multiple facts.

9. REMARKS
"remarks" is REQUIRED for the correct option only (empty for wrong).
Must be 2-3 sentences, minimum 25 words:
  FACT — one sentence stating the source fact.
  REASONING — one or two sentences explaining why that makes it correct.
Never just restate the answer. Never write a label like "Correct".

10. FINAL CHECK
Before returning JSON verify every question:
- answerable from source only
- names a specific concept, not a vague topic
- exactly one correct option
- 4 options in same category
- remark is 2-3 sentences, fact + reasoning
- no banned phrase appears anywhere
If any check fails, rewrite or drop the question.

11. HARD LIMIT
Return NO MORE THAN 3 questions total, regardless of chunk length.
If you have more candidates, keep only the 3 strongest.

"""

class Option(BaseModel):
    option: str
    remarks: str
    is_correct: bool

class MCQ(BaseModel):
    question: str
    difficulty: Literal["EASY", "MEDIUM", "HARD"]
    status: Literal["PENDING"] = "PENDING"
    options: list[Option] = Field(min_length=4, max_length=4)

class MCQList(BaseModel):
    mcqs: list[MCQ] = Field(max_length=4)

def generate_mcqs(chunk_text):
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=f"Generate UP TO 10 MCQs from this text.\n\n{chunk_text}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=MCQList,
        ),
    )
    return json.loads(response.text)["mcqs"]

with open("physics_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")

candidates = []
for i, chunk in enumerate(chunks):
    text = chunk["text"]
    print(f"[{i+1}/{len(chunks)}] id={chunk['id']} ({len(text)} chars)")
    try:
        mcqs = generate_mcqs(text)
        for q in mcqs:
            candidates.append({"chunk_id": chunk["id"], **q})
        print(f"{len(mcqs)} candidates" if mcqs else "   0 candidates")
    except Exception as e:
        print(f"   error: {e}")
    time.sleep(2)

with open("physics_mcqs.json", "w", encoding="utf-8") as f:
    json.dump(candidates, f, ensure_ascii=False, indent=2)

print(f"\nTotal candidates: {len(candidates)}")
print("Save to mcqs_json")

Total chunks: 44
[1/44] id=1 (1971 chars)
   3 candidates
[2/44] id=2 (1680 chars)
   3 candidates
[3/44] id=3 (1932 chars)
   3 candidates
[4/44] id=4 (1758 chars)
   3 candidates
[5/44] id=5 (1767 chars)
   2 candidates
[6/44] id=6 (1855 chars)
   3 candidates
[7/44] id=7 (1751 chars)
   3 candidates
[8/44] id=8 (1421 chars)
   3 candidates
[9/44] id=9 (1911 chars)
   3 candidates
[10/44] id=10 (1986 chars)
   3 candidates
[11/44] id=11 (1906 chars)
   3 candidates
[12/44] id=12 (1932 chars)
   3 candidates
[13/44] id=13 (1817 chars)
   3 candidates
[14/44] id=14 (1826 chars)
   3 candidates
[15/44] id=15 (1709 chars)
   3 candidates
[16/44] id=16 (1990 chars)
   3 candidates
[17/44] id=17 (1876 chars)
   3 candidates
[18/44] id=18 (1945 chars)
   1 candidates
[19/44] id=19 (1920 chars)
   3 candidates
[20/44] id=20 (1874 chars)
   3 candidates
[21/44] id=21 (1972 chars)
   3 candidates
[22/44] id=22 (1939 chars)
   3 candidates
[23/44] id=23 (1998 chars)
   3 candidates
[24/44] id=2